# Retrieval, Pembobotan, & Evaluasi
Notebook ini menyatukan tiga tahap yang saling terkait:
- **Struktur evaluasi**: bangun pasangan berlabel (positif=mirip, hard-negatif=tidak_mirip, easy-negatif=sampling) dan query retrieval (query→target di haystack).
- **Split level case (5-fold CV)**: tanpa kebocoran antar-merek; N kecil (55) → cross-validation.
- **Grid search bobot & threshold + evaluasi**: cari `w·JW + w·fonetik + w·semantik` terbaik, bandingkan hybrid vs single-method, lalu ukur retrieval (Recall@k/MRR/MAP) dan klasifikasi (P/R/F1/AUC).

Input dari Langkah C: `haystack_preprocessed_fonID.csv`, `haystack_emb.npy`, `haystack_emb_index.csv`, `gold_cases_categorized.csv`, `fitur_merek.py`, dan model semantik di Drive (`skripsi_merek/cc.id.100.bin`).


## 0. Setup & muat data

In [ ]:
!pip install rapidfuzz metaphone fasttext-wheel scikit-learn -q
import os, re, numpy as np, pandas as pd, itertools
from rapidfuzz import process
from rapidfuzz.distance import JaroWinkler
from metaphone import doublemetaphone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
np.random.seed(42)
print("Setup selesai.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 30.6 MB/s eta 0:00:00
Setup selesai.


In [ ]:
# Muat file dari Langkah C (upload bila belum ada)
butuh = ["haystack_preprocessed_fonID.csv","haystack_emb.npy","haystack_emb_index.csv",
         "gold_cases_categorized.csv","fitur_merek.py"]
if not all(os.path.exists(f) for f in butuh):
    from google.colab import files
    print("Upload file dari Langkah C:", [f for f in butuh if not os.path.exists(f)])
    files.upload()

import fitur_merek
from fitur_merek import jw_sim, phon_sim

hay  = pd.read_csv("haystack_preprocessed_fonID.csv", dtype=str).fillna("")
hay  = hay[hay["nama_base"].str.len()>0].reset_index(drop=True)
gold = pd.read_csv("gold_cases_categorized.csv", dtype=str).fillna("")
emb_index = pd.read_csv("haystack_emb_index.csv", dtype=str).fillna("")
emb = np.load("haystack_emb.npy")
print("Haystack:", hay.shape, "| emb:", emb.shape, "| gold:", gold.shape)

Upload file dari Langkah C: ['haystack_preprocessed_fonID.csv']


Saving haystack_preprocessed_fonID.csv to haystack_preprocessed_fonID.csv
Haystack: (13514, 11) | emb: (13514, 100) | gold: (95, 20)


## 1. Muat model semantik dari Drive
Model tereduksi (100-dim) yang disimpan Langkah C. Dipakai untuk skor semantik query.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
REDUCED_PATH = "/content/drive/MyDrive/skripsi_merek/cc.id.100.bin"
SEM_DIM = emb.shape[1]

sem_model = None
try:
    import fasttext
    sem_model = fasttext.load_model(REDUCED_PATH)
    print("Model semantik dimuat dari Drive.")
except Exception as e:
    print("Model semantik tidak dimuat:", repr(e))
    print(">> Skor semantik akan 0. Jalankan Langkah C dulu agar cc.id.100.bin ada di Drive.")

def sem_vec(text):
    if sem_model is None or not str(text).strip(): return None
    return sem_model.get_sentence_vector(str(text).replace("\n"," "))

def sem_sim(a, b):
    va, vb = sem_vec(a), sem_vec(b)
    if va is None or vb is None: return 0.0
    na, nb = np.linalg.norm(va), np.linalg.norm(vb)
    return float(np.dot(va, vb)/(na*nb)) if na and nb else 0.0

Mounted at /content/drive


Model semantik dimuat dari Drive.


## 2. (D) Bangun pasangan berlabel untuk klasifikasi
- **Positif**: pasangan `mirip` (set bermakna).
- **Hard-negatif**: pasangan `tidak_mirip` (set bermakna) — negatif paling berharga (kasus disengketakan tapi diputus tidak mirip).
- **Easy-negatif**: sampling merek acak dari haystack (jelas beda) → memberi cukup negatif untuk threshold/AUC.

`grup` menandai case induk agar split tidak bocor (easy-neg ikut grup positifnya).


In [ ]:
gold_eval = gold[gold["kategori_eval"]=="bermakna"].reset_index(drop=True)

rows = []
for i, r in gold_eval.iterrows():
    y = 1 if r["label_usulan"]=="mirip" else 0
    rows.append({"grup": f"case_{i}", "a": r["merek_a_base"], "b": r["merek_b_base"],
                 "a_ph": r["merek_a_phon"], "b_ph": r["merek_b_phon"],
                 "y": y, "jenis": "hard"})

N_EASY = 3
hay_u = hay[["nama_base","nama_phon"]].drop_duplicates("nama_base").reset_index(drop=True)
pos_rows = [x for x in rows if x["y"]==1]
for x in pos_rows:
    ambil = 0; tries = 0
    while ambil < N_EASY and tries < 30:
        c = hay_u.sample(1).iloc[0]; tries += 1
        if jw_sim(x["a"], c["nama_base"]) < 0.6:   # pastikan benar-benar beda
            rows.append({"grup": x["grup"], "a": x["a"], "b": c["nama_base"],
                         "a_ph": x["a_ph"], "b_ph": c["nama_phon"], "y": 0, "jenis": "easy"})
            ambil += 1

pairs = pd.DataFrame(rows)
pairs["jw"]   = pairs.apply(lambda r: jw_sim(r["a"], r["b"]), axis=1)
pairs["phon"] = pairs.apply(lambda r: phon_sim(r["a_ph"], r["b_ph"]), axis=1)
pairs["sem"]  = pairs.apply(lambda r: sem_sim(r["a"], r["b"]), axis=1)
print("Pasangan:", len(pairs),
      "| pos:", (pairs.y==1).sum(),
      "| neg hard:", ((pairs.y==0)&(pairs.jenis=='hard')).sum(),
      "| neg easy:", ((pairs.y==0)&(pairs.jenis=='easy')).sum())
print("\nRata-rata fitur per label:")
print(pairs.groupby("y")[["jw","phon","sem"]].mean().round(3).to_string())

Pasangan: 196 | pos: 47 | neg hard: 8 | neg easy: 141

Rata-rata fitur per label:
      jw   phon    sem
y                     
0  0.430  0.449  0.333
1  0.779  0.806  0.681


## 3. (E–F) Grid search bobot & threshold via 5-fold CV (level case)
Objektif: F1 (utamakan recall pada konteks pemeriksaan merek). Bobot `w1+w2+w3=1`, langkah 0.1 (66 kombinasi). Threshold di-scan bersama bobot, keduanya dipilih **di train**, diuji di **fold tersembunyi**.


In [ ]:
def grid_weights(step=0.1):
    r = [round(i*step, 4) for i in range(int(round(1/step))+1)]
    combos = []
    for w1 in r:
        for w2 in r:
            w3 = round(1 - w1 - w2, 4)
            if -1e-9 <= w3 <= 1+1e-9:
                combos.append((w1, w2, w3))
    return combos

WS = grid_weights(0.1)
THS = np.linspace(0.30, 0.95, 27)
Xf = pairs[["jw","phon","sem"]].values
y  = pairs["y"].values
groups = pairs["grup"].values

def eval_config(idx, w, th):
    s = w[0]*Xf[idx,0] + w[1]*Xf[idx,1] + w[2]*Xf[idx,2]
    pred = (s >= th).astype(int)
    return (f1_score(y[idx], pred, zero_division=0),
            precision_score(y[idx], pred, zero_division=0),
            recall_score(y[idx], pred, zero_division=0))

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fold_rows = []
for fold, (tr, te) in enumerate(sgkf.split(Xf, y, groups)):
    best = None
    for w in WS:
        for th in THS:
            f1,_,_ = eval_config(tr, w, th)
            if best is None or f1 > best[0]:
                best = (f1, w, th)
    _, w, th = best
    f1,pr,rc = eval_config(te, w, th)
    # AUC pada fold (skor hybrid)
    s_te = w[0]*Xf[te,0]+w[1]*Xf[te,1]+w[2]*Xf[te,2]
    try: auc = roc_auc_score(y[te], s_te)
    except: auc = float("nan")
    fold_rows.append({"fold":fold,"w1":w[0],"w2":w[1],"w3":w[2],"th":round(th,2),
                      "F1":round(f1,3),"Prec":round(pr,3),"Rec":round(rc,3),"AUC":round(auc,3)})

cv = pd.DataFrame(fold_rows)
print("=== Hasil per fold (HYBRID) ===")
print(cv.to_string(index=False))
print("\n=== Rata-rata CV (hybrid) ===")
for m in ["F1","Prec","Rec","AUC"]:
    print(f"  {m}: {cv[m].mean():.3f} ± {cv[m].std():.3f}")

=== Hasil per fold (HYBRID) ===
 fold  w1  w2  w3   th    F1  Prec   Rec   AUC
    0 0.5 0.2 0.3 0.57 0.875 0.875 0.875 0.962
    1 0.5 0.1 0.4 0.55 0.941 1.000 0.889 0.988
    2 0.5 0.1 0.4 0.57 0.737 0.778 0.700 0.953
    3 0.5 0.1 0.4 0.55 0.909 0.833 1.000 1.000
    4 0.6 0.1 0.3 0.57 0.842 0.889 0.800 0.935

=== Rata-rata CV (hybrid) ===
  F1: 0.861 ± 0.078
  Prec: 0.875 ± 0.082
  Rec: 0.853 ± 0.111
  AUC: 0.968 ± 0.026


## 4. Baseline single-method (untuk menjawab: apakah hybrid unggul?)

In [ ]:
def cv_single(col):
    f1s, recs = [], []
    for tr, te in sgkf.split(Xf, y, groups):
        best_th = max(THS, key=lambda th: f1_score(y[tr], (Xf[tr,col]>=th).astype(int), zero_division=0))
        pred = (Xf[te,col]>=best_th).astype(int)
        f1s.append(f1_score(y[te], pred, zero_division=0))
        recs.append(recall_score(y[te], pred, zero_division=0))
    return np.mean(f1s), np.mean(recs)

print("=== Perbandingan (rata-rata F1 / Recall CV) ===")
print(f"  JW saja       : F1 {cv_single(0)[0]:.3f} | Rec {cv_single(0)[1]:.3f}")
print(f"  Fonetik saja  : F1 {cv_single(1)[0]:.3f} | Rec {cv_single(1)[1]:.3f}")
print(f"  Semantik saja : F1 {cv_single(2)[0]:.3f} | Rec {cv_single(2)[1]:.3f}")
print(f"  HYBRID        : F1 {cv['F1'].mean():.3f} | Rec {cv['Rec'].mean():.3f}")

=== Perbandingan (rata-rata F1 / Recall CV) ===
  JW saja       : F1 0.839 | Rec 0.768
  Fonetik saja  : F1 0.740 | Rec 0.743
  Semantik saja : F1 0.685 | Rec 0.678
  HYBRID        : F1 0.861 | Rec 0.853


## 5. Bobot & threshold final (refit di seluruh data)
CV mengukur generalisasi; untuk konfigurasi yang "dipakai" (deployed), grid search diulang di seluruh pasangan. Ini bobot yang dipakai untuk retrieval di bawah.


In [ ]:
best = None
for w in WS:
    for th in THS:
        f1,_,_ = eval_config(np.arange(len(y)), w, th)
        if best is None or f1 > best[0]:
            best = (f1, w, th)
_, W_FINAL, TH_FINAL = best
print(f"Bobot final  : JW={W_FINAL[0]}, Fonetik={W_FINAL[1]}, Semantik={W_FINAL[2]}")
print(f"Threshold    : {TH_FINAL:.2f}")
print("\nInterpretasi: bobot mencerminkan kontribusi tiap dimensi. Bobot semantik kecil")
print("konsisten dgn Penjelasan Ps.21 (menekankan bunyi & tulisan, bukan makna).")

Bobot final  : JW=0.5, Fonetik=0.1, Semantik=0.4
Threshold    : 0.55

Interpretasi: bobot mencerminkan kontribusi tiap dimensi. Bobot semantik kecil
konsisten dgn Penjelasan Ps.21 (menekankan bunyi & tulisan, bukan makna).


## 6. Evaluasi RETRIEVAL (Recall@k, MRR, MAP)
Tiap query `mirip` (=merek_a) dicari terhadap seluruh haystack; dicek peringkat target (=merek_b). Basis pencarian = seluruh haystack valid (memuat semua target). Self-match (query=kandidat) dikecualikan.


In [ ]:
# precompute kode & embedding kandidat
cand_base = hay["nama_base"].tolist()
cand_dm   = hay["nama_phon"].apply(lambda t: "".join(doublemetaphone(x)[0] for x in t.split())).tolist()
# selaraskan embedding kandidat dgn urutan hay (via id)
id2row = {rid:i for i,rid in enumerate(emb_index["id"].tolist())}
cand_emb = np.vstack([emb[id2row[i]] if i in id2row else np.zeros(SEM_DIM) for i in hay["id"]]).astype("float32")
cand_emb_norm = cand_emb / (np.linalg.norm(cand_emb, axis=1, keepdims=True)+1e-9)

def retrieve(q_base, q_phon):
    qdm = "".join(doublemetaphone(x)[0] for x in q_phon.split())
    jw = process.cdist([q_base], cand_base, scorer=JaroWinkler.normalized_similarity)[0]/1.0
    ph = process.cdist([qdm], cand_dm, scorer=JaroWinkler.normalized_similarity)[0]/1.0
    qv = sem_vec(q_base)
    if qv is not None:
        qv = qv/(np.linalg.norm(qv)+1e-9); se = cand_emb_norm @ qv
    else:
        se = np.zeros(len(cand_base))
    return W_FINAL[0]*jw + W_FINAL[1]*ph + W_FINAL[2]*se

mirip = gold_eval[gold_eval["label_usulan"]=="mirip"].reset_index(drop=True)
ranks, tak_terjangkau = [], 0
for _, r in mirip.iterrows():
    score = retrieve(r["merek_a_base"], r["merek_a_phon"])
    order = np.argsort(-score)
    found = None
    for rank, i in enumerate(order, 1):
        if cand_base[i]==r["merek_b_base"] and cand_base[i]!=r["merek_a_base"]:
            found = rank; break
    if found is None: tak_terjangkau += 1
    else: ranks.append(found)

ranks = np.array(ranks)
print(f"Query mirip: {len(mirip)} | target terjangkau: {len(ranks)} | tak terjangkau: {tak_terjangkau}")
print("\n=== Metrik Retrieval ===")
for k in [1,5,10,20]:
    print(f"  Recall@{k}: {(ranks<=k).mean():.3f}")
print(f"  MRR      : {np.mean(1/ranks):.3f}")
print(f"  MAP      : {np.mean(1/ranks):.3f}  (1 relevan/query -> MAP=MRR)")
print(f"  Median rank target: {np.median(ranks):.0f}")

Query mirip: 47 | target terjangkau: 39 | tak terjangkau: 8

=== Metrik Retrieval ===
  Recall@1: 0.026
  Recall@5: 0.564
  Recall@10: 0.615
  Recall@20: 0.667
  MRR      : 0.282
  MAP      : 0.282  (1 relevan/query -> MAP=MRR)
  Median rank target: 3


## 7. Simpan hasil & ringkasan


In [ ]:
cv.to_csv("hasil_cv_hybrid.csv", index=False)
pairs.to_csv("pasangan_berlabel_fitur.csv", index=False, encoding="utf-8-sig")
ringkasan = {
    "n_pasangan": len(pairs), "n_query_mirip": len(mirip),
    "bobot_JW": W_FINAL[0], "bobot_fonetik": W_FINAL[1], "bobot_semantik": W_FINAL[2],
    "threshold": round(TH_FINAL,2),
    "F1_cv": round(cv["F1"].mean(),3), "Recall_cv": round(cv["Rec"].mean(),3),
    "AUC_cv": round(cv["AUC"].mean(),3),
    "Recall@10": round((ranks<=10).mean(),3), "MRR": round(np.mean(1/ranks),3),
}
pd.DataFrame([ringkasan]).to_csv("ringkasan_hasil.csv", index=False)
print("Tersimpan: hasil_cv_hybrid.csv, pasangan_berlabel_fitur.csv, ringkasan_hasil.csv")
print(ringkasan)

Tersimpan: hasil_cv_hybrid.csv, pasangan_berlabel_fitur.csv, ringkasan_hasil.csv
{'n_pasangan': 196, 'n_query_mirip': 47, 'bobot_JW': 0.5, 'bobot_fonetik': 0.1, 'bobot_semantik': 0.4, 'threshold': np.float64(0.55), 'F1_cv': np.float64(0.861), 'Recall_cv': np.float64(0.853), 'AUC_cv': np.float64(0.968), 'Recall@10': np.float64(0.615), 'MRR': np.float64(0.282)}


## 8. Ablasi (untuk BAB IV — bukti kontribusi tiap komponen)
Jalankan ulang seluruh pipeline dengan variasi berikut, lalu bandingkan `ringkasan_hasil.csv`:

1. **Normalisasi fonetik ID**: ganti input ke file `*_noFonID` (dari Langkah B dgn `AKTIFKAN_FONETIK_ID=False`).
2. **Semantik**: set bobot semantik nol (hybrid tekstual+fonetik saja) vs dengan semantik.
3. **Pretrained vs self-trained**: bandingkan sumber semantik.

Selisih metrik antar-variasi = bukti empiris kontribusi tiap komponen. Ini memperkuat pembahasan.

**Lanjut ke Langkah G–H:** error analysis (kasus salah tebak, hard-negative, reputasi_dominan yang dikeluarkan) dan penulisan BAB III–IV berdasarkan angka yang sudah keluar.
